# RandomEmbed Transform

Demonstrates `RandomEmbed`, a GPU batch transform that embeds images onto a
larger canvas with controllable positioning and configurable backgrounds.

**Operation:** Place an input image (C×H×W) at a random position on a larger
canvas (C×M×N), filling the rest with a configurable background.

**Key parameters:**
- `canvas_size`: int or (H, W) for the output canvas
- `x_range`, `y_range`: position in [0, 1] — 0=left/top, 0.5=center, 1=right/bottom
- `coords` / `coords_dict`: alternative positioning via explicit pixel coordinates
- `background`: `"zeros"`, `"mean"`, `"power_law"`
- `color_noise`: `True` (default) for independent per-channel noise, `False` for grayscale
- `mean` / `std`: optional normalisation — when `std` is provided, backgrounds are normalised as `(bg - mean) / std` to match normalised image pipelines
- `seed`: for reproducibility

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from slipstream import (
    SlipstreamDataset,
    SlipstreamLoader,
    DecodeRandomResizeShortCropLong,
    RandomEmbed,
    ToTorchImage,
    IMAGENET_MEAN,
    IMAGENET_STD,
)

dataset = SlipstreamDataset(
    remote_dir=LITDATA_VAL_PATH,
    decode_images=False,
)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def load_images(dataset, size=96, batch_size=8):
    """Load a batch of images decoded at the given size."""
    dec = DecodeRandomResizeShortCropLong(size=size, to_tensor=True, permute=True)
    loader = SlipstreamLoader(
        dataset, batch_size=batch_size, shuffle=False,
        pipelines={'image': [dec]},
        exclude_fields=['path'], verbose=False,
    )
    batch = next(iter(loader))
    loader.shutdown()
    # Convert to float [0, 1] for the transform
    return batch['image'].float() / 255.0


def show_embedded(images, n_cols=8, suptitle=None, row_labels=None,
                  mean=None, std=None):
    """Show a batch or list of batches of embedded images (CHW float tensors).

    If *mean* and *std* are provided (lists of per-channel values), images are
    denormalized before display:  img = img * std + mean.
    This allows correct visualisation of ImageNet-normalised tensors.
    """
    if isinstance(images, torch.Tensor) and images.ndim == 4:
        images = [images]
    n_rows = len(images)
    n_cols = min(n_cols, images[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 2.8 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, batch in enumerate(images):
        for c in range(n_cols):
            img = batch[c].clone()
            # Denormalize if mean/std provided
            if mean is not None and std is not None:
                for ch in range(img.shape[0]):
                    img[ch] = img[ch] * std[ch] + mean[ch]
            img = img.permute(1, 2, 0).clamp(0, 1).numpy()
            axes[r, c].imshow(img)
            axes[r, c].axis('off')
        if row_labels:
            axes[r, 0].set_ylabel(row_labels[r], fontsize=10, rotation=0,
                                   labelpad=60, va='center')
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


# Load our reference batch of 96x96 images
images = load_images(dataset, size=96, batch_size=8)
print(f"Loaded images: {images.shape}, dtype={images.dtype}, range=[{images.min():.2f}, {images.max():.2f}]")

## 1. Basic Center Embed (Default)

Default `x_range=0.5, y_range=0.5` centers the image on the canvas.
Background is zeros (black).

In [ ]:
embed = RandomEmbed(canvas_size=224)
out = embed(images)
print(f"Input: {images.shape} \u2192 Output: {out.shape}")
show_embedded(out, suptitle='Center embed: 96\u00d796 on 224\u00d7224 canvas (zeros background)')

## 2. Position Control (x_range / y_range)

Sliding `x_range` from 0 (left) to 1 (right). For these square images on a
square canvas, `y_range` works identically on the vertical axis.

In [ ]:
positions = [0.0, 0.25, 0.5, 0.75, 1.0]
rows = []
for xp in positions:
    embed = RandomEmbed(canvas_size=224, x_range=xp, y_range=xp)
    rows.append(embed(images))

show_embedded(
    rows,
    row_labels=[f'x=y={p}' for p in positions],
    suptitle='Position control: sliding from top-left (0) to bottom-right (1)',
)

## 3. Random Positions

`x_range=(0, 1), y_range=(0, 1)` with different seeds.
Each row shows the same images embedded at different random positions.

In [ ]:
seeds = [42, 43, 44]
rows = []
for seed in seeds:
    embed = RandomEmbed(canvas_size=224, x_range=(0, 1), y_range=(0, 1), seed=seed)
    rows.append(embed(images))

show_embedded(
    rows,
    row_labels=[f'seed={s}' for s in seeds],
    suptitle='Random positions: same images, different seeds',
)

## 4. Background Comparison: zeros vs mean vs power_law

Same images at the same position (center), with three different background types.

In [ ]:
bg_configs = [
    ('zeros', dict(background='zeros')),
    ('mean (ImageNet)', dict(background='mean', mean=[0.485, 0.456, 0.406])),
    ('power_law (α=0)', dict(background='power_law', alpha_range=0.0, seed=42)),
    ('power_law (α=1)', dict(background='power_law', alpha_range=1.0, seed=42)),
    ('power_law (α=2)', dict(background='power_law', alpha_range=2.0, seed=42)),
]

rows = []
labels = []
for label, kwargs in bg_configs:
    embed = RandomEmbed(canvas_size=224, **kwargs)
    rows.append(embed(images))
    labels.append(label)

show_embedded(rows, row_labels=labels,
              suptitle='Background comparison (center embed)')

## 5. Power-Law Noise Alpha Sweep

α=0 (white noise), α=1 (pink noise), α=2 (Brownian noise).
Higher alpha produces smoother, more spatially correlated noise.

In [ ]:
alphas = [0.0, 0.5, 1.0, 1.5, 2.0]
rows = []
labels = []
for alpha in alphas:
    embed = RandomEmbed(
        canvas_size=224, background='power_law',
        alpha_range=alpha, seed=42,
    )
    rows.append(embed(images))
    labels.append(f'α={alpha}')

show_embedded(rows, row_labels=labels,
              suptitle='Power-law noise: α=0 (white) → α=2 (Brownian)')

## 6. Color vs Grayscale Noise

`color_noise=True` (default) generates independent noise per R, G, B channel,
producing colorful backgrounds that make it harder for models to segment the
image from the background by color alone.  `color_noise=False` duplicates a
single noise field across all channels (grayscale).

In [ ]:
color_configs = [
    ('color (default)', dict(color_noise=True)),
    ('grayscale', dict(color_noise=False)),
]

rows = []
labels = []
for label, kwargs in color_configs:
    embed = RandomEmbed(
        canvas_size=224, background='power_law',
        alpha_range=1.5, seed=42, **kwargs,
    )
    rows.append(embed(images))
    labels.append(label)

show_embedded(rows, row_labels=labels,
              suptitle='Color noise (independent channels) vs grayscale noise')

## 7. Coords Mode

Provide specific `(x, y)` center-pixel coordinates. Each image samples one
coordinate from the list. Positions near the edge clip the image.

In [ ]:
# Single fixed coordinate: top-left quadrant
embed_tl = RandomEmbed(canvas_size=224, coords=[(72, 72)])
# Single fixed coordinate: bottom-right quadrant
embed_br = RandomEmbed(canvas_size=224, coords=[(152, 152)])
# Multiple coordinates: randomly pick one per image
embed_multi = RandomEmbed(canvas_size=224, coords=[(56, 56), (168, 168), (112, 56), (56, 168)], seed=42)

show_embedded(
    [embed_tl(images), embed_br(images), embed_multi(images)],
    row_labels=['center=(72,72)', 'center=(152,152)', 'multi coords'],
    suptitle='Coords mode: explicit center coordinates',
)

## 8. Multi-Size Embedding

Decode images at different sizes (64, 96, 128, 160), embed all on a 224×224 canvas.
Smaller images have more space around them.

In [ ]:
sizes = [64, 96, 128, 160]
rows = []
labels = []
for sz in sizes:
    imgs = load_images(dataset, size=sz, batch_size=8)
    embed = RandomEmbed(
        canvas_size=224,
        x_range=(0, 1), y_range=(0, 1),
        background='power_law', alpha_range=1.5,
        seed=42,
    )
    rows.append(embed(imgs))
    labels.append(f'{sz}×{sz}')

show_embedded(rows, row_labels=labels,
              suptitle='Multi-size embed on 224×224 (power-law background)')

## 9. SSL Replay Demo

Call `embed(view1)` then `embed.apply_last(view2)` — both views are placed
at the **same position** (stored from `before_call`), but the background
noise differs between the two calls.

In [ ]:
# Load two different "views" (different decoder seeds = different crops)
view1 = load_images(dataset, size=96, batch_size=8)
view2 = load_images(dataset, size=96, batch_size=8)

embed = RandomEmbed(
    canvas_size=224,
    x_range=(0, 1), y_range=(0, 1),
    background='power_law', alpha_range=1.0,
    seed=42,
)

out1 = embed(view1)              # before_call + apply_last
out2 = embed.apply_last(view2)   # reuses stored positions

# Print positions to verify they're identical
params = embed.last_params()
print('Positions (shared across both views):')
for i in range(min(4, len(params['xs']))):
    print(f"  image {i}: x={params['xs'][i].item()}, y={params['ys'][i].item()}")

show_embedded(
    [out1, out2],
    row_labels=['view 1', 'view 2 (replay)'],
    suptitle='SSL replay: same positions, different images & backgrounds',
)

## 10. Normalised-Image Pipeline

When images are normalised (mean-subtracted, std-divided), pass `mean` and
`std` to `RandomEmbed` so the background is normalised the same way:
`(bg - mean) / std`.  The `show_embedded` helper's `mean`/`std` arguments
denormalize for display.

In [ ]:
from slipstream import Normalize, IMAGENET_MEAN, IMAGENET_STD

# Normalize images to ImageNet stats
norm = Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
images_norm = norm(images)
print(f"Normalised images: range=[{images_norm.min():.2f}, {images_norm.max():.2f}]")

# Embed with mean/std so backgrounds are normalised to match
bg_configs_norm = [
    ('zeros', dict(background='zeros', mean=IMAGENET_MEAN, std=IMAGENET_STD)),
    ('mean', dict(background='mean', mean=IMAGENET_MEAN, std=IMAGENET_STD)),
    ('power_law', dict(background='power_law', alpha_range=1.5,
                       mean=IMAGENET_MEAN, std=IMAGENET_STD, seed=42)),
]

rows = []
labels = []
for label, kwargs in bg_configs_norm:
    embed = RandomEmbed(canvas_size=224, **kwargs)
    rows.append(embed(images_norm))
    labels.append(label)

# Pass mean/std to show_embedded so it denormalizes for display
show_embedded(rows, row_labels=labels,
              mean=IMAGENET_MEAN, std=IMAGENET_STD,
              suptitle='Normalised images: embed + denormalize for display')

## Summary

| Parameter | Effect |
|-----------|--------|
| `canvas_size=224` | 224×224 output canvas |
| `x_range=(0, 1)` | Full horizontal jitter |
| `y_range=0.5` | Vertically centered |
| `coords=[(x,y), ...]` | Explicit center coordinates |
| `coords_dict={(H,W): [...]}` | Per-size coordinates |
| `background="zeros"` | Black background |
| `background="mean"` | Constant fill (ImageNet mean) |
| `background="power_law"` | 1/f^α colored noise |
| `color_noise=True` | Independent noise per channel (default) |
| `color_noise=False` | Grayscale (same noise all channels) |
| `mean=..., std=...` | Normalise background to match normalised images |
| `seed=42` | Reproducible positions |